# ∂-NO Benchmark: Experiments

## Supplementary Material

> **Note:** This notebook is provided for illustration purposes only.  
> Full code and reproducible experiments will be available on GitHub upon acceptance.
>
> Results may vary with hardware configurations.  
> **Tip:** Lower batch sizes generally yield better metrics but require longer training time.
> **Datasets:** Please place them in ./data folder

---

### Overview

This notebook is a standalone supplementary material comparing baseline neural operators with their ∂-augmented variants.

### Datasets

| Dataset | PDE | Domain | Resolution | Train/Test |
|---------|-----|--------|------------|------------|
| Burgers (ν=0.1) | Viscous Burgers | 1D | 8192 → 1024 | 1000/200 |
| Darcy | Elliptic (steady-state) | 2D | 421×421 → 85×85 | 1000/200 |
| Navier-Stokes (ν=10⁻⁵) | Incompressible turbulence | 2D+T | 64×64×20 | 1000/200 |

### Models

| Model | Description | ∂-Augmentation |
|-------|-------------|----------------|
| FNO | Fourier Neural Operator | ∂-FNO |
| Transolver | Attention-based (SOTA) | ∂-Transolver |

### ∂-NO Configuration

| Setting | 1D Problems | 2D Problems |
|---------|-------------|-------------|
| Derivative features | $\mathfrak{D}_x^{\beta_1} u$, $\mathfrak{D}_x^{\beta_2} u$ | $\mathfrak{D}_x^{\beta_{x,1}}$, $\mathfrak{D}_x^{\beta_{x,2}}$, $\mathfrak{D}_y^{\beta_{y,1}}$, $\mathfrak{D}_y^{\beta_{y,2}}$, mixed |
| Initialization | β₁=1.0, β₂=2.0 | Same per direction |
| GL kernel size | K=16 | K=16 |

### Training Configuration

| Parameter | Value |
|-----------|-------|
| Optimizer | AdamW |
| LR (backbone) | 10⁻³ |
| LR (β parameters) | 10⁻² (10× higher) |
| Batch size | 8 (FNO) / 4 (Transolver) |
| Normalizer | UnitGaussianNormalizer (Darcy) |
| NS training | Autoregressive |

### Experiment Matrix

| | FNO | ∂-FNO | Transolver | ∂-Transolver |
|---|:---:|:---:|:---:|:---:|
| Burgers (ν=0.1) | ✓ | ✓ | ✓ | ✓ |
| Darcy | ✓ | ✓ | ✓ | ✓ |
| Navier-Stokes | ✓ | ✓ | ✓ | ✓ |

In [23]:
# @title 1. IMPORTS
# ══════════════════════════════════════════════════════════════════════════════════════════
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
import os
import time
import csv
import glob
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Union, Any, Callable
from dataclasses import dataclass, field

try:
    import scipy.io
    HAS_SCIPY = True
except ImportError:
    HAS_SCIPY = False

try:
    import gdown
    HAS_GDOWN = True
except ImportError:
    HAS_GDOWN = False


# ══════════════════════════════════════════════════════════════════════════════════════════

In [25]:
# @title 2. GOOGLE DRIVE IDS
# ══════════════════════════════════════════════════════════════════════════════════════════
GDRIVE_IDS = {
    'burgers_r10':   '16a8od4vidbiNR3WtaBPCSZ0T3moxjhYe',
    'darcy_421':     '1Z1uxG9R8AdAGJprG5STcphysjm56_0Jf',
    'ns_v1e5':       '1qO46jjKooiymGCjtfKxb9fUfa74fc68Z',
}


# ══════════════════════════════════════════════════════════════════════════════════════════

In [26]:
# @title 3. CONFIGURATION DATACLASS
# ══════════════════════════════════════════════════════════════════════════════════════════
@dataclass
class SpectraConfig:
    """Configuration for DELNO-Spectra v7."""

    # ─── Dataset ─────────────────────────────────────────────────────────────────
    dataset: str = 'burgers_r10'
    n_train: int = 1000
    n_test: int = 200
    resolution: int = 1024

    # ─── Model ───────────────────────────────────────────────────────────────────
    model_type: str = 'fno'  # 'fno' or 'transolver'

    # ─── DELNO-Spectra Features ──────────────────────────────────────────────────
    max_order: int = 2
    m_max: float = 2.0
    include_cross: bool = False
    include_grid: bool = True
    K: int = 16  # GL kernel size

    # ─── FNO Backbone ────────────────────────────────────────────────────────────
    width: int = 64
    modes: int = 12
    n_layers: int = 4

    # ─── Transolver ──────────────────────────────────────────────────────────────
    n_heads: int = 8
    slice_num: int = 32
    mlp_ratio: int = 2
    dropout: float = 0.0

    # ─── Training ────────────────────────────────────────────────────────────────
    epochs: int = 5
    batch_size: int = 8
    lr: float = 1e-3
    lr_beta: float = 1e-2
    lr_heff: float = 1e-3
    weight_decay: float = 1e-5
    scheduler: str = 'onecycle'  # 'onecycle', 'step', 'cosine'
    pct_start: float = 0.3
    step_size: int = 100
    gamma: float = 0.5
    grad_clip: float = 1.0

    # ─── Logging ─────────────────────────────────────────────────────────────────
    log_dir: str = './logs'
    log_every: int = 10
    eval_every: int = 100

    # ─── Misc ────────────────────────────────────────────────────────────────────
    seed: int = 42
    device: str = 'cuda'
    preset: str = 'thuml'


# ══════════════════════════════════════════════════════════════════════════════════════════

In [27]:
# @title 4. MODULAR CONFIGURATION SYSTEM
# ══════════════════════════════════════════════════════════════════════════════════════════
# Dataset-specific parameters
DATASET_PARAMS = {
    'burgers_r10': {
        'n_train': 1000, 'n_test': 200, 'resolution': 1024,
    },
    'burgers_r1000': {
        'n_train': 1000, 'n_test': 200, 'resolution': 1024,
    },
    'darcy': {
        'n_train': 1000, 'n_test': 200, 'resolution': 85,
    },
    'ns_v1e3': {
        'n_train': 1000, 'n_test': 200, 'resolution': 64,
    },
    'ns_v1e5': {
        'n_train': 1000, 'n_test': 200, 'resolution': 64,
    },
}

# Model-specific parameters
MODEL_PARAMS = {
    'fno': {
        'burgers_r10':  {'width': 64,  'modes': 12, 'n_layers': 4},
        'burgers_r1000': {'width': 64,  'modes': 12, 'n_layers': 4},
        'darcy':         {'width': 64,  'modes': 12, 'n_layers': 4},  # original uses 64
        'ns_v1e3':       {'width': 64,  'modes': 12, 'n_layers': 4},
        'ns_v1e5':       {'width': 64,  'modes': 12, 'n_layers': 4},
    },
    'transolver': {
        'burgers_r10':  {'width': 128, 'n_layers': 8, 'n_heads': 8, 'slice_num': 64},
        'burgers_r1000': {'width': 128, 'n_layers': 8, 'n_heads': 8, 'slice_num': 64},
        'darcy':         {'width': 128, 'n_layers': 8, 'n_heads': 8, 'slice_num': 64},
        'ns_v1e3':       {'width': 256, 'n_layers': 8, 'n_heads': 8, 'slice_num': 32},
        'ns_v1e5':       {'width': 256, 'n_layers': 8, 'n_heads': 8, 'slice_num': 32},
    },
}

# Training preset parameters
PRESET_PARAMS = {
    'original': {
        'epochs': 500, 'batch_size': 20, 'lr': 1e-3, 'weight_decay': 1e-4,
        'scheduler': 'step', 'step_size': 100, 'gamma': 0.5,
    },
    'thuml': {
        'epochs': 500, 'batch_size': 8, 'lr': 1e-3, 'weight_decay': 1e-5,
        'scheduler': 'onecycle', 'pct_start': 0.3,
    },
    'custom': {
        'epochs': 1000, 'batch_size': 8, 'lr': 1e-3, 'weight_decay': 1e-5,
        'scheduler': 'step', 'step_size': 200, 'gamma': 0.5,
    },
}

# Dataset-specific overrides per preset
DATASET_PRESET_OVERRIDES = {
    ('darcy', 'thuml'): {'width': 128, 'batch_size': 4},
    ('ns_v1e3', 'thuml'): {'lr': 5e-4, 'batch_size': 20, 'scheduler': 'step'},
    ('ns_v1e3', 'original'): {'lr': 1e-3},
    ('ns_v1e3', 'custom'): {'lr': 5e-4, 'batch_size': 20},
    ('ns_v1e5', 'thuml'): {'lr': 5e-4, 'batch_size': 20, 'scheduler': 'step'},
    ('ns_v1e5', 'original'): {'lr': 1e-3},
    ('ns_v1e5', 'custom'): {'lr': 5e-4, 'batch_size': 20},
    ('burgers_r10', 'original'): {'modes': 16},
    ('burgers_r1000', 'original'): {'modes': 16},
}

# Transolver-specific overrides
TRANSOLVER_PRESET_OVERRIDES = {
    ('ns_v1e3', 'thuml'): {'batch_size': 2},
    ('ns_v1e5', 'thuml'): {'batch_size': 2},
    ('ns_v1e3', 'custom'): {'batch_size': 2},
    ('ns_v1e5', 'custom'): {'batch_size': 2},
}


def get_config(dataset: str, model: str, preset: str) -> SpectraConfig:
    """
    Get configuration by composing dataset, model, and preset parameters.

    Args:
        dataset: 'burgers_r100', 'burgers_r1000', 'darcy', 'ns_v1e3', 'ns_v1e5'
        model: 'fno', 'transolver'
        preset: 'original', 'thuml', 'custom'

    Returns:
        SpectraConfig with merged parameters
    """
    if dataset not in DATASET_PARAMS:
        raise ValueError(f"Unknown dataset: {dataset}")
    if model not in MODEL_PARAMS:
        raise ValueError(f"Unknown model: {model}")
    if preset not in PRESET_PARAMS:
        raise ValueError(f"Unknown preset: {preset}")

    # Start with defaults
    params = {'dataset': dataset, 'model_type': model, 'preset': preset}

    # Add dataset params
    params.update(DATASET_PARAMS[dataset])

    # Add model params
    params.update(MODEL_PARAMS[model][dataset])

    # Add preset params
    params.update(PRESET_PARAMS[preset])

    # Apply dataset-preset overrides
    key = (dataset, preset)
    if key in DATASET_PRESET_OVERRIDES:
        params.update(DATASET_PRESET_OVERRIDES[key])

    # Apply transolver-specific overrides
    if model == 'transolver' and key in TRANSOLVER_PRESET_OVERRIDES:
        params.update(TRANSOLVER_PRESET_OVERRIDES[key])

    return SpectraConfig(**params)


def list_configs() -> List[str]:
    """List all available configuration names."""
    configs = []
    for dataset in DATASET_PARAMS.keys():
        for model in MODEL_PARAMS.keys():
            for preset in PRESET_PARAMS.keys():
                configs.append(f"{dataset}_{model}_{preset}")
    return configs


# Legacy compatibility
CONFIGS = {name: lambda d=name.rsplit('_', 2)[0], m=name.rsplit('_', 2)[1], p=name.rsplit('_', 2)[2]: get_config(d, m, p)
           for name in list_configs()}


# ══════════════════════════════════════════════════════════════════════════════════════════

In [28]:
# @title 5.UTILITIES
# ══════════════════════════════════════════════════════════════════════════════════════════
def count_params(model: nn.Module) -> int:
    """Count trainable parameters."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def rel_l2_loss(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    """Relative L2 loss."""
    pred_flat = pred.reshape(pred.shape[0], -1)
    target_flat = target.reshape(target.shape[0], -1)
    return (torch.norm(pred_flat - target_flat, dim=-1) /
            (torch.norm(target_flat, dim=-1) + 1e-8)).mean()


def l2_loss(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    """MSE loss."""
    return F.mse_loss(pred, target)


def setup_device(cfg: SpectraConfig) -> torch.device:
    """Setup compute device."""
    if cfg.device == 'cuda' and torch.cuda.is_available():
        torch.backends.cudnn.benchmark = True
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        device = torch.device('cuda')
        print(f"Device: {torch.cuda.get_device_name(0)}")
    else:
        device = torch.device('cpu')
        print("Device: CPU")
    return device


def set_seed(seed: int):
    """Set random seeds for reproducibility."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class UnitGaussianNormalizer:
    """Normalize to zero mean and unit variance."""
    def __init__(self, x: torch.Tensor, eps: float = 1e-5):
        self.mean = x.mean(dim=0, keepdim=True)
        self.std = x.std(dim=0, keepdim=True)
        self.eps = eps

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        return (x - self.mean) / (self.std + self.eps)

    def decode(self, x: torch.Tensor) -> torch.Tensor:
        return x * (self.std + self.eps) + self.mean

    def to(self, device: torch.device) -> 'UnitGaussianNormalizer':
        self.mean = self.mean.to(device)
        self.std = self.std.to(device)
        return self


def softplus_inverse(x: float) -> float:
    """Inverse of softplus: log(exp(x) - 1)."""
    return math.log(math.exp(x) - 1) if x > 0 else -10.0


# ══════════════════════════════════════════════════════════════════════════════════════════

In [29]:
# @title 6. CSV LOGGER
# ══════════════════════════════════════════════════════════════════════════════════════════
class CSVLogger:
    """
    CSV Logger for training metrics.

    Logs: epoch, train_loss, test_loss, rel_l2, lr, time_s, betas, h_effs, scales

    For 1D max_order=2: 2 betas, 2 h_effs, 2 scales
    For 2D max_order=2: 6 betas, 5 h_effs, 5 scales
    """
    def __init__(self, log_dir: str, experiment_name: str, cfg: SpectraConfig, is_2d: bool = False):
        os.makedirs(log_dir, exist_ok=True)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.csv_path = os.path.join(log_dir, f"{experiment_name}_{timestamp}.csv")
        self.cfg = cfg
        self.is_2d = is_2d

        self.fields = ['epoch', 'train_loss', 'test_loss', 'rel_l2', 'lr', 'time_s']

        # Determine number of betas and h_effs
        if is_2d and cfg.max_order == 2:
            self.n_betas, self.n_heffs = 6, 5
            self.deriv_names = ['Dx', 'Dy', 'Dxx', 'Dyy', 'Dxy_x', 'Dxy_y']
            self.heff_names = ['Dx', 'Dy', 'Dxx', 'Dyy', 'Dxy']
        elif is_2d and cfg.max_order == 1:
            self.n_betas, self.n_heffs = 2, 2
            self.deriv_names = ['Dx', 'Dy']
            self.heff_names = ['Dx', 'Dy']
        else:
            self.n_betas = self.n_heffs = cfg.max_order
            self.deriv_names = [f'D{i+1}' for i in range(cfg.max_order)]
            self.heff_names = self.deriv_names

        for name in self.deriv_names:
            self.fields.append(f'beta_{name}')
        for name in self.heff_names:
            self.fields.extend([f'h_eff_{name}', f'scale_{name}'])

        with open(self.csv_path, 'w', newline='') as f:
            csv.DictWriter(f, fieldnames=self.fields).writeheader()

        print(f"  Logging to: {self.csv_path}")

    def log(self, epoch: int, train_loss: float, test_loss: float, rel_l2: float,
            lr: float, time_s: float, spectra_params: Optional[Dict] = None):
        """Log a single epoch's metrics."""
        row = {
            'epoch': epoch, 'train_loss': f'{train_loss:.6f}',
            'test_loss': f'{test_loss:.6f}', 'rel_l2': f'{rel_l2:.6f}',
            'lr': f'{lr:.2e}', 'time_s': f'{time_s:.1f}'
        }

        if spectra_params and 'betas' in spectra_params:
            betas = spectra_params['betas']
            h_effs = spectra_params['h_effs']
            scales = spectra_params['scales']

            for i, name in enumerate(self.deriv_names):
                if i < len(betas):
                    row[f'beta_{name}'] = f'{betas[i]:.4f}'

            for i, name in enumerate(self.heff_names):
                if i < len(h_effs):
                    row[f'h_eff_{name}'] = f'{h_effs[i]:.6f}'
                    row[f'scale_{name}'] = f'{scales[i]:.4f}'
        else:
            for name in self.deriv_names:
                row[f'beta_{name}'] = ''
            for name in self.heff_names:
                row[f'h_eff_{name}'] = row[f'scale_{name}'] = ''

        with open(self.csv_path, 'a', newline='') as f:
            csv.DictWriter(f, fieldnames=self.fields).writerow(row)

    def close(self, summary: Dict = None):
        if summary:
            print(f"  Final: rel_l2={summary.get('best_rel_l2', 0)*100:.3f}%")


# ══════════════════════════════════════════════════════════════════════════════════════════

In [30]:
# @title 7. GRÜNWALD-LETNIKOV FRACTIONAL DERIVATIVE
# ══════════════════════════════════════════════════════════════════════════════════════════
def gl_weights(beta: torch.Tensor, K: int) -> torch.Tensor:
    """GL weights via stable cumulative product."""
    device, dtype = beta.device, beta.dtype
    k = torch.arange(1, K, dtype=dtype, device=device)
    ratios = (k - 1 - beta) / k
    cumprods = torch.cumprod(ratios, dim=0)
    return torch.cat([torch.ones(1, dtype=dtype, device=device), cumprods])


def gl_diff_1d_raw(u: torch.Tensor, beta: torch.Tensor, K: int = 16,
                   padding: str = 'circular') -> torch.Tensor:
    """1D GL fractional derivative WITHOUT scaling."""
    squeeze = u.dim() == 2
    if squeeze:
        u = u.unsqueeze(1)
    B, C, N = u.shape
    K = min(K, N)

    w = gl_weights(beta, K)

    # Backward difference
    u_back = F.pad(u, (K-1, 0), mode=padding)
    k_back = w.flip(0).view(1, 1, K).expand(C, 1, K)
    d_back = F.conv1d(u_back, k_back, groups=C)[:, :, :N]

    # Forward difference
    u_fwd = F.pad(u, (0, K-1), mode=padding)
    k_fwd = w.view(1, 1, K).expand(C, 1, K)
    d_fwd = F.conv1d(u_fwd, k_fwd, groups=C)[:, :, :N]

    result = 0.5 * (d_back + torch.cos(math.pi * beta) * d_fwd)
    return result.squeeze(1) if squeeze else result


def gl_diff_2d_raw(u: torch.Tensor, beta_x: torch.Tensor, beta_y: torch.Tensor,
                   K: int = 16, padding: str = 'circular') -> torch.Tensor:
    """2D GL fractional derivative."""
    B, H, W = u.shape

    if beta_x.abs() > 1e-8:
        u_flat = u.reshape(B * H, W)
        u_flat = gl_diff_1d_raw(u_flat, beta_x, K, padding)
        u = u_flat.reshape(B, H, W)

    if beta_y.abs() > 1e-8:
        u_t = u.permute(0, 2, 1).reshape(B * W, H)
        u_t = gl_diff_1d_raw(u_t, beta_y, K, padding)
        u = u_t.reshape(B, W, H).permute(0, 2, 1)

    return u


# ══════════════════════════════════════════════════════════════════════════════════════════

In [31]:
# @title 8. SPECTRA FRACTIONAL DERIVATIVE MODULES
# ══════════════════════════════════════════════════════════════════════════════════════════
class SpectraFracDeriv1D(nn.Module):
    """
    1D Fractional derivative with learnable β and h_eff.

    Scale formula: scale = h_eff^(-β)
    h_eff initialized from: h = 2/modes (spectral grid spacing)
    """
    def __init__(self, beta_init: float, m_max: float, modes: int, K: int = 16):
        super().__init__()
        self.m_max = m_max
        self.K = K
        self.modes = modes

        # Learnable β (clamped to [0, m_max])
        self.beta_raw = nn.Parameter(torch.tensor(beta_init))

        # Learnable h_eff initialized from modes: h = 2/modes
        h_init = 2.0 / modes
        self.h_eff_raw = nn.Parameter(torch.tensor(softplus_inverse(h_init)))

    @property
    def beta(self) -> torch.Tensor:
        return torch.clamp(self.beta_raw, 0.0, self.m_max)

    @property
    def h_eff(self) -> torch.Tensor:
        return F.softplus(self.h_eff_raw)

    @property
    def scale(self) -> torch.Tensor:
        return torch.pow(self.h_eff, -self.beta)

    def forward(self, u: torch.Tensor) -> torch.Tensor:
        d_raw = gl_diff_1d_raw(u, self.beta, self.K)
        return self.scale * d_raw

    def get_params(self) -> Dict[str, float]:
        return {'beta': self.beta.item(), 'h_eff': self.h_eff.item(), 'scale': self.scale.item()}


class SpectraFracDeriv2D(nn.Module):
    """
    2D Fractional derivative with learnable β_x, β_y and h_eff.

    Scale formula: scale = h_x^(-β_x) * h_y^(-β_y)
    For same modes in both directions: scale = (2/modes)^(-(β_x + β_y))
    """
    def __init__(self, beta_init_x: float, beta_init_y: float, m_max: float,
                 modes_x: int, modes_y: int, K: int = 16):
        super().__init__()
        self.m_max = m_max
        self.K = K
        self.modes_x = modes_x
        self.modes_y = modes_y

        self.beta_raw_x = nn.Parameter(torch.tensor(beta_init_x))
        self.beta_raw_y = nn.Parameter(torch.tensor(beta_init_y))

        # h_eff for x and y directions (can differ if modes differ)
        h_init_x = 2.0 / modes_x
        h_init_y = 2.0 / modes_y
        self.h_eff_raw_x = nn.Parameter(torch.tensor(softplus_inverse(h_init_x)))
        self.h_eff_raw_y = nn.Parameter(torch.tensor(softplus_inverse(h_init_y)))

    @property
    def beta_x(self) -> torch.Tensor:
        return torch.clamp(self.beta_raw_x, 0.0, self.m_max)

    @property
    def beta_y(self) -> torch.Tensor:
        return torch.clamp(self.beta_raw_y, 0.0, self.m_max)

    @property
    def h_eff_x(self) -> torch.Tensor:
        return F.softplus(self.h_eff_raw_x)

    @property
    def h_eff_y(self) -> torch.Tensor:
        return F.softplus(self.h_eff_raw_y)

    @property
    def scale(self) -> torch.Tensor:
        """scale = h_x^(-β_x) * h_y^(-β_y)"""
        scale_x = torch.pow(self.h_eff_x, -self.beta_x) if self.beta_x.abs() > 1e-8 else torch.tensor(1.0)
        scale_y = torch.pow(self.h_eff_y, -self.beta_y) if self.beta_y.abs() > 1e-8 else torch.tensor(1.0)
        return scale_x * scale_y

    def forward(self, u: torch.Tensor) -> torch.Tensor:
        d_raw = gl_diff_2d_raw(u, self.beta_x, self.beta_y, self.K)
        return self.scale * d_raw

    def get_params(self) -> Dict[str, float]:
        return {
            'beta_x': self.beta_x.item(), 'beta_y': self.beta_y.item(),
            'h_eff_x': self.h_eff_x.item(), 'h_eff_y': self.h_eff_y.item(),
            'scale': self.scale.item()
        }


# ══════════════════════════════════════════════════════════════════════════════════════════

In [32]:
# @title 9. SPECTRA FEATURES LAYERS
# ══════════════════════════════════════════════════════════════════════════════════════════
class SpectraFeatures1D(nn.Module):
    """
    1D Spectra feature layer.

    For max_order=2: D1 (β=1.0), D2 (β=2.0) → 2 betas, 2 h_effs
    """
    def __init__(self, cfg: SpectraConfig):
        super().__init__()
        self.max_order = cfg.max_order
        self.include_cross = cfg.include_cross
        self.include_grid = cfg.include_grid
        self.modes = cfg.modes

        # D1 init at β=1.0, D2 init at β=2.0, etc.
        self.derivs = nn.ModuleList([
            SpectraFracDeriv1D(float(order), cfg.m_max, cfg.modes, cfg.K)
            for order in range(1, cfg.max_order + 1)
        ])
        self.deriv_names = [f'D{order}' for order in range(1, cfg.max_order + 1)]

    @property
    def n_channels(self) -> int:
        n = 1 if self.include_grid else 0
        n += 1 + self.max_order
        if self.include_cross:
            n += self.max_order
        return n

    def forward(self, u: torch.Tensor) -> torch.Tensor:
        B, N = u.shape
        features = []

        if self.include_grid:
            x = torch.linspace(0, 1, N, device=u.device, dtype=u.dtype).unsqueeze(0).expand(B, -1)
            features.append(x)

        features.append(u)
        derivs_computed = [d(u) for d in self.derivs]
        features.extend(derivs_computed)

        if self.include_cross:
            features.extend([u * d for d in derivs_computed])

        return torch.stack(features, dim=-1)

    def get_params(self) -> Dict[str, Any]:
        betas, h_effs, scales = [], [], []
        for deriv in self.derivs:
            p = deriv.get_params()
            betas.append(p['beta'])
            h_effs.append(p['h_eff'])
            scales.append(p['scale'])
        return {'betas': betas, 'h_effs': h_effs, 'scales': scales}

    def beta_parameters(self) -> List[nn.Parameter]:
        return [d.beta_raw for d in self.derivs]

    def heff_parameters(self) -> List[nn.Parameter]:
        return [d.h_eff_raw for d in self.derivs]


class SpectraFeatures2D(nn.Module):
    """
    2D Spectra feature layer.

    For max_order=2:
        - Dx (β_x=1.0), Dy (β_y=1.0)  → 2 betas, 2 h_effs
        - Dxx (β_x=2.0), Dyy (β_y=2.0) → 2 betas, 2 h_effs
        - Dxy (β_x=1.0, β_y=1.0)       → 2 betas, 1 h_eff (shared)

    Total: 6 betas, 5 derivative operators (5 h_effs)
    """
    def __init__(self, cfg: SpectraConfig):
        super().__init__()
        self.max_order = cfg.max_order
        self.include_cross = cfg.include_cross
        self.include_grid = cfg.include_grid
        self.modes = cfg.modes

        self.derivs = nn.ModuleList()
        self.deriv_names = []

        # Order 1: Dx (β=1,0), Dy (β=0,1)
        if cfg.max_order >= 1:
            self.derivs.append(SpectraFracDeriv2D(1.0, 0.0, cfg.m_max, cfg.modes, cfg.modes, cfg.K))
            self.deriv_names.append('Dx')
            self.derivs.append(SpectraFracDeriv2D(0.0, 1.0, cfg.m_max, cfg.modes, cfg.modes, cfg.K))
            self.deriv_names.append('Dy')

        # Order 2: Dxx (β=2,0), Dyy (β=0,2), Dxy (β=1,1)
        if cfg.max_order >= 2:
            self.derivs.append(SpectraFracDeriv2D(2.0, 0.0, cfg.m_max, cfg.modes, cfg.modes, cfg.K))
            self.deriv_names.append('Dxx')
            self.derivs.append(SpectraFracDeriv2D(0.0, 2.0, cfg.m_max, cfg.modes, cfg.modes, cfg.K))
            self.deriv_names.append('Dyy')
            self.derivs.append(SpectraFracDeriv2D(1.0, 1.0, cfg.m_max, cfg.modes, cfg.modes, cfg.K))
            self.deriv_names.append('Dxy')

    @property
    def n_channels(self) -> int:
        n = 2 if self.include_grid else 0
        n += 1 + len(self.derivs)
        if self.include_cross:
            n += len(self.derivs)
        return n

    def forward(self, u: torch.Tensor) -> torch.Tensor:
        B, H, W = u.shape
        device = u.device
        features = []

        if self.include_grid:
            x = torch.linspace(0, 1, H, device=device, dtype=u.dtype)
            y = torch.linspace(0, 1, W, device=device, dtype=u.dtype)
            gx, gy = torch.meshgrid(x, y, indexing='ij')
            features.extend([gx.unsqueeze(0).expand(B, -1, -1), gy.unsqueeze(0).expand(B, -1, -1)])

        features.append(u)
        derivs_computed = [d(u) for d in self.derivs]
        features.extend(derivs_computed)

        if self.include_cross:
            features.extend([u * d for d in derivs_computed])

        return torch.stack(features, dim=-1)

    def get_params(self) -> Dict[str, Any]:
        """
        Returns 6 betas, 5 h_effs, 5 scales for max_order=2.

        Betas: [β_Dx, β_Dy, β_Dxx, β_Dyy, β_Dxy_x, β_Dxy_y]
        H_effs/Scales: one per derivative operator [Dx, Dy, Dxx, Dyy, Dxy]
        """
        betas, h_effs, scales = [], [], []

        for deriv, name in zip(self.derivs, self.deriv_names):
            p = deriv.get_params()
            h_effs.append(p['h_eff_x'] if p['beta_x'] > 1e-8 else p['h_eff_y'])
            scales.append(p['scale'])

            if name == 'Dxy':
                betas.extend([p['beta_x'], p['beta_y']])
            elif name in ['Dx', 'Dxx']:
                betas.append(p['beta_x'])
            else:
                betas.append(p['beta_y'])

        return {'betas': betas, 'h_effs': h_effs, 'scales': scales}

    def beta_parameters(self) -> List[nn.Parameter]:
        """Returns 6 parameters for max_order=2."""
        params = []
        for d, name in zip(self.derivs, self.deriv_names):
            if name == 'Dxy':
                params.extend([d.beta_raw_x, d.beta_raw_y])
            elif name in ['Dx', 'Dxx']:
                params.append(d.beta_raw_x)
            else:
                params.append(d.beta_raw_y)
        return params

    def heff_parameters(self) -> List[nn.Parameter]:
        """Returns h_eff parameters (active ones only)."""
        params = []
        for d, name in zip(self.derivs, self.deriv_names):
            if name == 'Dxy':
                params.extend([d.h_eff_raw_x, d.h_eff_raw_y])
            elif name in ['Dx', 'Dxx']:
                params.append(d.h_eff_raw_x)
            else:
                params.append(d.h_eff_raw_y)
        return params


# ══════════════════════════════════════════════════════════════════════════════════════════

In [33]:
# @title 10. SPECTRAL CONVOLUTION LAYERS
# ══════════════════════════════════════════════════════════════════════════════════════════
class SpectralConv1d(nn.Module):
    """1D Spectral convolution."""
    def __init__(self, in_channels: int, out_channels: int, modes: int):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.modes = modes
        self.scale = 1 / (in_channels * out_channels)
        self.weights = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, modes, dtype=torch.cfloat))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, N = x.shape
        x_ft = torch.fft.rfft(x, dim=-1)
        out_ft = torch.zeros(B, self.out_channels, N // 2 + 1, dtype=torch.cfloat, device=x.device)
        modes = min(self.modes, N // 2 + 1)
        out_ft[:, :, :modes] = torch.einsum('bix,iox->box', x_ft[:, :, :modes], self.weights[:, :, :modes])
        return torch.fft.irfft(out_ft, n=N, dim=-1)


class SpectralConv2d(nn.Module):
    """2D Spectral convolution."""
    def __init__(self, in_channels: int, out_channels: int, modes1: int, modes2: int):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.modes1 = modes1
        self.modes2 = modes2
        self.scale = 1 / (in_channels * out_channels)
        self.weights1 = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, modes1, modes2, dtype=torch.cfloat))
        self.weights2 = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, modes1, modes2, dtype=torch.cfloat))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        x_ft = torch.fft.rfft2(x, dim=(-2, -1))
        out_ft = torch.zeros(B, self.out_channels, H, W // 2 + 1, dtype=torch.cfloat, device=x.device)
        m1, m2 = min(self.modes1, H // 2), min(self.modes2, W // 2 + 1)
        out_ft[:, :, :m1, :m2] = torch.einsum('bixy,ioxy->boxy', x_ft[:, :, :m1, :m2], self.weights1[:, :, :m1, :m2])
        out_ft[:, :, -m1:, :m2] = torch.einsum('bixy,ioxy->boxy', x_ft[:, :, -m1:, :m2], self.weights2[:, :, :m1, :m2])
        return torch.fft.irfft2(out_ft, s=(H, W), dim=(-2, -1))


# ══════════════════════════════════════════════════════════════════════════════════════════

In [34]:
# @title 11. FNO MODELS
# ══════════════════════════════════════════════════════════════════════════════════════════
class FNO1d(nn.Module):
    """1D Fourier Neural Operator."""
    def __init__(self, cfg: SpectraConfig):
        super().__init__()
        self.lift = nn.Linear(2, cfg.width)  # x, u
        self.convs = nn.ModuleList([SpectralConv1d(cfg.width, cfg.width, cfg.modes) for _ in range(cfg.n_layers)])
        self.ws = nn.ModuleList([nn.Conv1d(cfg.width, cfg.width, 1) for _ in range(cfg.n_layers)])
        self.proj = nn.Sequential(nn.Linear(cfg.width, 128), nn.GELU(), nn.Linear(128, 1))

    def forward(self, u: torch.Tensor) -> torch.Tensor:
        B, N = u.shape
        x_coord = torch.linspace(0, 1, N, device=u.device).unsqueeze(0).expand(B, -1)
        x = self.lift(torch.stack([x_coord, u], dim=-1)).permute(0, 2, 1)

        for i, (conv, w) in enumerate(zip(self.convs, self.ws)):
            x1 = conv(x)
            x2 = w(x)
            x = F.gelu(x1 + x2) if i < len(self.convs) - 1 else x1 + x2

        return self.proj(x.permute(0, 2, 1)).squeeze(-1)


class FNO2d(nn.Module):
    """2D Fourier Neural Operator."""
    def __init__(self, cfg: SpectraConfig):
        super().__init__()
        self.lift = nn.Linear(3, cfg.width)  # x, y, u
        self.convs = nn.ModuleList([SpectralConv2d(cfg.width, cfg.width, cfg.modes, cfg.modes) for _ in range(cfg.n_layers)])
        self.ws = nn.ModuleList([nn.Conv2d(cfg.width, cfg.width, 1) for _ in range(cfg.n_layers)])
        self.proj = nn.Sequential(nn.Linear(cfg.width, 128), nn.GELU(), nn.Linear(128, 1))

    def forward(self, u: torch.Tensor) -> torch.Tensor:
        B, H, W = u.shape
        device = u.device
        gx = torch.linspace(0, 1, H, device=device).view(1, H, 1).expand(B, H, W)
        gy = torch.linspace(0, 1, W, device=device).view(1, 1, W).expand(B, H, W)

        x = self.lift(torch.stack([gx, gy, u], dim=-1)).permute(0, 3, 1, 2)

        for i, (conv, w) in enumerate(zip(self.convs, self.ws)):
            x1 = conv(x)
            x2 = w(x)
            x = F.gelu(x1 + x2) if i < len(self.convs) - 1 else x1 + x2

        return self.proj(x.permute(0, 2, 3, 1)).squeeze(-1)


class FNO2d_AR(nn.Module):
    """2D FNO for autoregressive temporal prediction."""
    def __init__(self, cfg: SpectraConfig, T_in: int = 10):
        super().__init__()
        self.T_in = T_in
        self.lift = nn.Linear(T_in + 2, cfg.width)
        self.convs = nn.ModuleList([SpectralConv2d(cfg.width, cfg.width, cfg.modes, cfg.modes) for _ in range(cfg.n_layers)])
        self.ws = nn.ModuleList([nn.Conv2d(cfg.width, cfg.width, 1) for _ in range(cfg.n_layers)])
        self.proj = nn.Sequential(nn.Linear(cfg.width, 128), nn.GELU(), nn.Linear(128, 1))

    def forward(self, u_history: torch.Tensor) -> torch.Tensor:
        # u_history: [B, T_in, H, W]
        B, T, H, W = u_history.shape
        device = u_history.device

        gx = torch.linspace(0, 1, H, device=device).view(1, 1, H, 1).expand(B, 1, H, W)
        gy = torch.linspace(0, 1, W, device=device).view(1, 1, 1, W).expand(B, 1, H, W)

        x = torch.cat([u_history, gx, gy], dim=1).permute(0, 2, 3, 1)  # [B, H, W, T+2]
        x = self.lift(x).permute(0, 3, 1, 2)  # [B, width, H, W]

        for i, (conv, w) in enumerate(zip(self.convs, self.ws)):
            x1 = conv(x)
            x2 = w(x)
            x = F.gelu(x1 + x2) if i < len(self.convs) - 1 else x1 + x2

        return self.proj(x.permute(0, 2, 3, 1)).squeeze(-1)


# ══════════════════════════════════════════════════════════════════════════════════════════

In [35]:
# @title 12. DEL-FNO MODELS
# ══════════════════════════════════════════════════════════════════════════════════════════
class DELFNO1d(nn.Module):
    """1D DEL-FNO with learnable fractional derivatives."""
    def __init__(self, cfg: SpectraConfig):
        super().__init__()
        self.features = SpectraFeatures1D(cfg)
        self.lift = nn.Linear(self.features.n_channels, cfg.width)
        self.convs = nn.ModuleList([SpectralConv1d(cfg.width, cfg.width, cfg.modes) for _ in range(cfg.n_layers)])
        self.ws = nn.ModuleList([nn.Conv1d(cfg.width, cfg.width, 1) for _ in range(cfg.n_layers)])
        self.proj = nn.Sequential(nn.Linear(cfg.width, 128), nn.GELU(), nn.Linear(128, 1))

    def forward(self, u: torch.Tensor) -> torch.Tensor:
        x = self.features(u)  # [B, N, n_channels]
        x = self.lift(x).permute(0, 2, 1)

        for i, (conv, w) in enumerate(zip(self.convs, self.ws)):
            x1 = conv(x)
            x2 = w(x)
            x = F.gelu(x1 + x2) if i < len(self.convs) - 1 else x1 + x2

        return self.proj(x.permute(0, 2, 1)).squeeze(-1)

    def get_params(self): return self.features.get_params()
    def beta_parameters(self): return self.features.beta_parameters()
    def heff_parameters(self): return self.features.heff_parameters()


class DELFNO2d(nn.Module):
    """2D DEL-FNO."""
    def __init__(self, cfg: SpectraConfig):
        super().__init__()
        self.features = SpectraFeatures2D(cfg)
        self.lift = nn.Linear(self.features.n_channels, cfg.width)
        self.convs = nn.ModuleList([SpectralConv2d(cfg.width, cfg.width, cfg.modes, cfg.modes) for _ in range(cfg.n_layers)])
        self.ws = nn.ModuleList([nn.Conv2d(cfg.width, cfg.width, 1) for _ in range(cfg.n_layers)])
        self.proj = nn.Sequential(nn.Linear(cfg.width, 128), nn.GELU(), nn.Linear(128, 1))

    def forward(self, u: torch.Tensor) -> torch.Tensor:
        x = self.features(u)  # [B, H, W, n_channels]
        x = self.lift(x).permute(0, 3, 1, 2)

        for i, (conv, w) in enumerate(zip(self.convs, self.ws)):
            x1 = conv(x)
            x2 = w(x)
            x = F.gelu(x1 + x2) if i < len(self.convs) - 1 else x1 + x2

        return self.proj(x.permute(0, 2, 3, 1)).squeeze(-1)

    def get_params(self): return self.features.get_params()
    def beta_parameters(self): return self.features.beta_parameters()
    def heff_parameters(self): return self.features.heff_parameters()


class DELFNO2d_AR(nn.Module):
    """2D DEL-FNO for autoregressive prediction."""
    def __init__(self, cfg: SpectraConfig, T_in: int = 10):
        super().__init__()
        self.T_in = T_in
        self.features = SpectraFeatures2D(cfg)
        in_dim = T_in + self.features.n_channels - 1 + 2  # history + features (minus u) + grid

        self.lift = nn.Linear(in_dim, cfg.width)
        self.convs = nn.ModuleList([SpectralConv2d(cfg.width, cfg.width, cfg.modes, cfg.modes) for _ in range(cfg.n_layers)])
        self.ws = nn.ModuleList([nn.Conv2d(cfg.width, cfg.width, 1) for _ in range(cfg.n_layers)])
        self.proj = nn.Sequential(nn.Linear(cfg.width, 128), nn.GELU(), nn.Linear(128, 1))

    def forward(self, u_history: torch.Tensor) -> torch.Tensor:
        B, T, H, W = u_history.shape
        device = u_history.device

        # Features from last timestep
        feat = self.features(u_history[:, -1])  # [B, H, W, n_feat]

        # Grid
        gx = torch.linspace(0, 1, H, device=device).view(1, H, 1, 1).expand(B, H, W, 1)
        gy = torch.linspace(0, 1, W, device=device).view(1, 1, W, 1).expand(B, H, W, 1)

        # Combine: history + derivative features (skip u) + grid
        x = torch.cat([u_history.permute(0, 2, 3, 1), feat[:, :, :, 1:], gx, gy], dim=-1)
        x = self.lift(x).permute(0, 3, 1, 2)

        for i, (conv, w) in enumerate(zip(self.convs, self.ws)):
            x1 = conv(x)
            x2 = w(x)
            x = F.gelu(x1 + x2) if i < len(self.convs) - 1 else x1 + x2

        return self.proj(x.permute(0, 2, 3, 1)).squeeze(-1)

    def get_params(self): return self.features.get_params()
    def beta_parameters(self): return self.features.beta_parameters()
    def heff_parameters(self): return self.features.heff_parameters()


# ══════════════════════════════════════════════════════════════════════════════════════════

In [36]:
# @title 13. PHYSICS-ATTENTION (thuml-compatible)
# ══════════════════════════════════════════════════════════════════════════════════════════
class PhysicsAttention1D(nn.Module):
    """Physics-Attention for 1D data (thuml-compatible)."""
    def __init__(self, hidden_dim: int, n_heads: int, slice_num: int, mlp_ratio: int = 2, dropout: float = 0.0):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.n_heads = n_heads
        self.slice_num = slice_num
        self.head_dim = hidden_dim // n_heads
        inner_dim = self.head_dim * n_heads
        self.scale = self.head_dim ** -0.5

        self.softmax = nn.Softmax(dim=-1)
        self.dropout_layer = nn.Dropout(dropout)
        self.temperature = nn.Parameter(torch.ones(1, n_heads, 1, 1) * 0.5)

        self.in_project_x = nn.Linear(hidden_dim, inner_dim)
        self.in_project_fx = nn.Linear(hidden_dim, inner_dim)
        self.in_project_slice = nn.Linear(self.head_dim, slice_num)
        nn.init.orthogonal_(self.in_project_slice.weight)

        self.to_q = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.to_k = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.to_v = nn.Linear(self.head_dim, self.head_dim, bias=False)

        self.to_out = nn.Sequential(nn.Linear(inner_dim, hidden_dim), nn.Dropout(dropout))
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * mlp_ratio), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim * mlp_ratio, hidden_dim), nn.Dropout(dropout))
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, N, C = x.shape

        # Slice
        fx_mid = self.in_project_fx(x).reshape(B, N, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
        x_mid = self.in_project_x(x).reshape(B, N, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
        slice_weights = self.softmax(self.in_project_slice(x_mid) / self.temperature)
        slice_norm = slice_weights.sum(2)
        slice_token = torch.einsum("bhnc,bhng->bhgc", fx_mid, slice_weights) / (slice_norm[:, :, :, None] + 1e-5)

        # Attention
        q, k, v = self.to_q(slice_token), self.to_k(slice_token), self.to_v(slice_token)
        attn = self.softmax(torch.matmul(q, k.transpose(-1, -2)) * self.scale)
        out_slice_token = torch.matmul(self.dropout_layer(attn), v)

        # Deslice
        out_x = torch.einsum("bhgc,bhng->bhnc", out_slice_token, slice_weights)
        out_x = out_x.permute(0, 2, 1, 3).reshape(B, N, self.n_heads * self.head_dim)

        x = self.norm1(x + self.to_out(out_x))
        return self.norm2(x + self.mlp(x))


class PhysicsAttention2D(nn.Module):
    """Physics-Attention for 2D data (thuml-compatible)."""
    def __init__(self, hidden_dim: int, n_heads: int, slice_num: int, mlp_ratio: int = 2, dropout: float = 0.0):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.n_heads = n_heads
        self.slice_num = slice_num
        self.head_dim = hidden_dim // n_heads
        inner_dim = self.head_dim * n_heads
        self.scale = self.head_dim ** -0.5

        self.softmax = nn.Softmax(dim=-1)
        self.dropout_layer = nn.Dropout(dropout)
        self.temperature = nn.Parameter(torch.ones(1, n_heads, 1, 1) * 0.5)

        self.in_project_x = nn.Linear(hidden_dim, inner_dim)
        self.in_project_fx = nn.Linear(hidden_dim, inner_dim)
        self.in_project_slice = nn.Linear(self.head_dim, slice_num)
        nn.init.orthogonal_(self.in_project_slice.weight)

        self.to_q = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.to_k = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.to_v = nn.Linear(self.head_dim, self.head_dim, bias=False)

        self.to_out = nn.Sequential(nn.Linear(inner_dim, hidden_dim), nn.Dropout(dropout))
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * mlp_ratio), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim * mlp_ratio, hidden_dim), nn.Dropout(dropout))
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, N, C = x.shape

        fx_mid = self.in_project_fx(x).reshape(B, N, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
        x_mid = self.in_project_x(x).reshape(B, N, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
        slice_weights = self.softmax(self.in_project_slice(x_mid) / torch.clamp(self.temperature, 0.1, 5))
        slice_norm = slice_weights.sum(2)
        slice_token = torch.einsum("bhnc,bhng->bhgc", fx_mid, slice_weights) / (slice_norm[:, :, :, None] + 1e-5)

        q, k, v = self.to_q(slice_token), self.to_k(slice_token), self.to_v(slice_token)
        attn = self.softmax(torch.matmul(q, k.transpose(-1, -2)) * self.scale)
        out_slice_token = torch.matmul(self.dropout_layer(attn), v)

        out_x = torch.einsum("bhgc,bhng->bhnc", out_slice_token, slice_weights)
        out_x = out_x.permute(0, 2, 1, 3).reshape(B, N, self.n_heads * self.head_dim)

        x = self.norm1(x + self.to_out(out_x))
        return self.norm2(x + self.mlp(x))


class TransolverBlock(nn.Module):
    """Single Transolver block."""
    def __init__(self, hidden_dim: int, n_heads: int, slice_num: int, mlp_ratio: int, dropout: float, dim: int = 1):
        super().__init__()
        AttnClass = PhysicsAttention1D if dim == 1 else PhysicsAttention2D
        self.attn = AttnClass(hidden_dim, n_heads, slice_num, mlp_ratio, dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.attn(x)


# ══════════════════════════════════════════════════════════════════════════════════════════

In [37]:
# @title 14. TRANSOLVER MODELS
# ══════════════════════════════════════════════════════════════════════════════════════════
class Transolver1D(nn.Module):
    """1D Transolver."""
    def __init__(self, cfg: SpectraConfig):
        super().__init__()
        self.lift = nn.Linear(2, cfg.width)
        self.blocks = nn.ModuleList([
            TransolverBlock(cfg.width, cfg.n_heads, cfg.slice_num, cfg.mlp_ratio, cfg.dropout, dim=1)
            for _ in range(cfg.n_layers)])
        self.proj = nn.Sequential(nn.Linear(cfg.width, 128), nn.GELU(), nn.Linear(128, 1))

    def forward(self, u: torch.Tensor) -> torch.Tensor:
        B, N = u.shape
        x_coord = torch.linspace(0, 1, N, device=u.device).unsqueeze(0).expand(B, -1)
        x = self.lift(torch.stack([x_coord, u], dim=-1))
        for block in self.blocks:
            x = block(x)
        return self.proj(x).squeeze(-1)


class Transolver2D(nn.Module):
    """2D Transolver."""
    def __init__(self, cfg: SpectraConfig):
        super().__init__()
        self.lift = nn.Linear(3, cfg.width)
        self.blocks = nn.ModuleList([
            TransolverBlock(cfg.width, cfg.n_heads, cfg.slice_num, cfg.mlp_ratio, cfg.dropout, dim=2)
            for _ in range(cfg.n_layers)])
        self.proj = nn.Sequential(nn.Linear(cfg.width, 128), nn.GELU(), nn.Linear(128, 1))

    def forward(self, u: torch.Tensor) -> torch.Tensor:
        B, H, W = u.shape
        device = u.device
        gx = torch.linspace(0, 1, H, device=device).view(1, H, 1).expand(B, H, W)
        gy = torch.linspace(0, 1, W, device=device).view(1, 1, W).expand(B, H, W)
        x = self.lift(torch.stack([gx, gy, u], dim=-1).reshape(B, H * W, 3))
        for block in self.blocks:
            x = block(x)
        return self.proj(x).squeeze(-1).reshape(B, H, W)


class Transolver2D_AR(nn.Module):
    """2D Transolver for autoregressive prediction."""
    def __init__(self, cfg: SpectraConfig, T_in: int = 10):
        super().__init__()
        self.T_in = T_in
        self.lift = nn.Linear(T_in + 2, cfg.width)
        self.blocks = nn.ModuleList([
            TransolverBlock(cfg.width, cfg.n_heads, cfg.slice_num, cfg.mlp_ratio, cfg.dropout, dim=2)
            for _ in range(cfg.n_layers)])
        self.proj = nn.Sequential(nn.Linear(cfg.width, 128), nn.GELU(), nn.Linear(128, 1))

    def forward(self, u_history: torch.Tensor) -> torch.Tensor:
        B, T, H, W = u_history.shape
        device = u_history.device
        gx = torch.linspace(0, 1, H, device=device).view(1, H, 1).expand(B, H, W)
        gy = torch.linspace(0, 1, W, device=device).view(1, 1, W).expand(B, H, W)
        grid = torch.stack([gx, gy], dim=1)

        x = torch.cat([u_history, grid], dim=1).permute(0, 2, 3, 1).reshape(B, H * W, T + 2)
        x = self.lift(x)
        for block in self.blocks:
            x = block(x)
        return self.proj(x).squeeze(-1).reshape(B, H, W)


# ══════════════════════════════════════════════════════════════════════════════════════════

In [38]:
# @title 15. DEL-TRANSOLVER MODELS
# ══════════════════════════════════════════════════════════════════════════════════════════
class DELTransolver1D(nn.Module):
    """1D DEL-Transolver."""
    def __init__(self, cfg: SpectraConfig):
        super().__init__()
        self.features = SpectraFeatures1D(cfg)
        self.lift = nn.Linear(self.features.n_channels, cfg.width)
        self.blocks = nn.ModuleList([
            TransolverBlock(cfg.width, cfg.n_heads, cfg.slice_num, cfg.mlp_ratio, cfg.dropout, dim=1)
            for _ in range(cfg.n_layers)])
        self.proj = nn.Sequential(nn.Linear(cfg.width, 128), nn.GELU(), nn.Linear(128, 1))

    def forward(self, u: torch.Tensor) -> torch.Tensor:
        x = self.lift(self.features(u))
        for block in self.blocks:
            x = block(x)
        return self.proj(x).squeeze(-1)

    def get_params(self): return self.features.get_params()
    def beta_parameters(self): return self.features.beta_parameters()
    def heff_parameters(self): return self.features.heff_parameters()


class DELTransolver2D(nn.Module):
    """2D DEL-Transolver."""
    def __init__(self, cfg: SpectraConfig):
        super().__init__()
        self.features = SpectraFeatures2D(cfg)
        self.lift = nn.Linear(self.features.n_channels, cfg.width)
        self.blocks = nn.ModuleList([
            TransolverBlock(cfg.width, cfg.n_heads, cfg.slice_num, cfg.mlp_ratio, cfg.dropout, dim=2)
            for _ in range(cfg.n_layers)])
        self.proj = nn.Sequential(nn.Linear(cfg.width, 128), nn.GELU(), nn.Linear(128, 1))

    def forward(self, u: torch.Tensor) -> torch.Tensor:
        B, H, W = u.shape
        x = self.lift(self.features(u).reshape(B, H * W, -1))
        for block in self.blocks:
            x = block(x)
        return self.proj(x).squeeze(-1).reshape(B, H, W)

    def get_params(self): return self.features.get_params()
    def beta_parameters(self): return self.features.beta_parameters()
    def heff_parameters(self): return self.features.heff_parameters()


class DELTransolver2D_AR(nn.Module):
    """2D DEL-Transolver for autoregressive prediction."""
    def __init__(self, cfg: SpectraConfig, T_in: int = 10):
        super().__init__()
        self.T_in = T_in
        self.features = SpectraFeatures2D(cfg)
        in_dim = T_in + self.features.n_channels - 1 + 2

        self.lift = nn.Linear(in_dim, cfg.width)
        self.blocks = nn.ModuleList([
            TransolverBlock(cfg.width, cfg.n_heads, cfg.slice_num, cfg.mlp_ratio, cfg.dropout, dim=2)
            for _ in range(cfg.n_layers)])
        self.proj = nn.Sequential(nn.Linear(cfg.width, 128), nn.GELU(), nn.Linear(128, 1))

    def forward(self, u_history: torch.Tensor) -> torch.Tensor:
        B, T, H, W = u_history.shape
        device = u_history.device

        feat = self.features(u_history[:, -1])
        gx = torch.linspace(0, 1, H, device=device).view(1, H, 1, 1).expand(B, H, W, 1)
        gy = torch.linspace(0, 1, W, device=device).view(1, 1, W, 1).expand(B, H, W, 1)

        x = torch.cat([u_history.permute(0, 2, 3, 1), feat[:, :, :, 1:], gx, gy], dim=-1)
        x = self.lift(x.reshape(B, H * W, -1))
        for block in self.blocks:
            x = block(x)
        return self.proj(x).squeeze(-1).reshape(B, H, W)

    def get_params(self): return self.features.get_params()
    def beta_parameters(self): return self.features.beta_parameters()
    def heff_parameters(self): return self.features.heff_parameters()


# ══════════════════════════════════════════════════════════════════════════════════════════

In [39]:
# @title 16. MODEL FACTORY
# ══════════════════════════════════════════════════════════════════════════════════════════
def create_model(cfg: SpectraConfig, use_del: bool = False, T_in: int = 10) -> nn.Module:
    """Create model based on configuration."""
    dataset = cfg.dataset.lower()
    model_type = cfg.model_type.lower()
    is_1d = 'burgers' in dataset
    is_temporal = 'ns' in dataset

    if model_type == 'fno':
        if is_1d:
            return DELFNO1d(cfg) if use_del else FNO1d(cfg)
        elif is_temporal:
            return DELFNO2d_AR(cfg, T_in) if use_del else FNO2d_AR(cfg, T_in)
        else:
            return DELFNO2d(cfg) if use_del else FNO2d(cfg)
    elif model_type == 'transolver':
        if is_1d:
            return DELTransolver1D(cfg) if use_del else Transolver1D(cfg)
        elif is_temporal:
            return DELTransolver2D_AR(cfg, T_in) if use_del else Transolver2D_AR(cfg, T_in)
        else:
            return DELTransolver2D(cfg) if use_del else Transolver2D(cfg)
    else:
        raise ValueError(f"Unknown model type: {model_type}")


# ══════════════════════════════════════════════════════════════════════════════════════════

In [40]:
# @title 17. DATA LOADERS
# ══════════════════════════════════════════════════════════════════════════════════════════
def download_gdrive(file_id: str, output: str):
    """Download file from Google Drive."""
    if not HAS_GDOWN:
        raise ImportError("Install gdown: pip install gdown")
    gdown.download(f'https://drive.google.com/uc?id={file_id}', output, quiet=False)


def load_burgers(cfg: SpectraConfig, device: torch.device, data_dir: str = './data') -> Dict:
    """Load Burgers R10 data."""
    os.makedirs(data_dir, exist_ok=True)

    # Try to find existing file
    patterns = [
        f'{data_dir}/burgers_data_R10.mat',
        f'{data_dir}/Burgers_R10*.mat',
        f'{data_dir}/*burgers*.mat',
    ]

    filepath = None
    for pattern in patterns:
        if '*' in pattern:
            files = glob.glob(pattern)
            if files:
                filepath = files[0]
                break
        elif os.path.exists(pattern):
            filepath = pattern
            break

    if filepath is None and GDRIVE_IDS.get('burgers_r10'):
        print("Downloading Burgers R10 data...")
        filepath = f'{data_dir}/burgers_r10.mat'
        download_gdrive(GDRIVE_IDS['burgers_r10'], filepath)

    if filepath is None:
        raise FileNotFoundError("Could not find Burgers R10 data")

    # Load
    try:
        import h5py
        with h5py.File(filepath, 'r') as f:
            a = np.array(f['a']).T.astype(np.float32)
            u = np.array(f['u']).T.astype(np.float32)
    except:
        mat = scipy.io.loadmat(filepath)
        a, u = mat['a'].astype(np.float32), mat['u'].astype(np.float32)

    a, u = torch.from_numpy(a), torch.from_numpy(u)

    # Subsample
    N_orig = a.shape[-1]
    if cfg.resolution < N_orig:
        step = N_orig // cfg.resolution
        a = a[:, ::step][:, :cfg.resolution]
        u = u[:, ::step][:, :cfg.resolution]

    return {
        'a_train': a[:cfg.n_train].to(device),
        'u_train': u[:cfg.n_train].to(device),
        'a_test': a[cfg.n_train:cfg.n_train + cfg.n_test].to(device),
        'u_test': u[cfg.n_train:cfg.n_train + cfg.n_test].to(device),
        'N': a.shape[-1],
    }


def load_burgers_r1000(cfg: SpectraConfig, device: torch.device, data_dir: str = './data') -> Dict:
    """Load Burgers R1000 data."""
    os.makedirs(data_dir, exist_ok=True)

    patterns = [
        f'{data_dir}/burgers_v1000*.mat',
        f'{data_dir}/Burgers*1000*.mat',
        f'{data_dir}/burgers_r1000*.mat',
    ]

    filepath = None
    for pattern in patterns:
        files = glob.glob(pattern)
        if files:
            filepath = files[0]
            break

    if filepath is None and GDRIVE_IDS.get('burgers_r1000'):
        print("Downloading Burgers R1000 data...")
        filepath = f'{data_dir}/burgers_r1000.mat'
        download_gdrive(GDRIVE_IDS['burgers_r1000'], filepath)

    if filepath is None:
        raise FileNotFoundError("Could not find Burgers R1000 data")

    # Load
    try:
        import h5py
        with h5py.File(filepath, 'r') as f:
            keys = list(f.keys())
            if 'input' in f:
                a = np.array(f['input']).astype(np.float32)
                u = np.array(f['output']).astype(np.float32)
                if u.ndim == 3:
                    u = u[:, -1, :] if u.shape[1] < u.shape[2] else u[-1, :, :]
            else:
                a = np.array(f['a']).T.astype(np.float32)
                u = np.array(f['u']).T.astype(np.float32)
    except:
        mat = scipy.io.loadmat(filepath)
        a, u = mat.get('a', mat.get('input')).astype(np.float32), mat.get('u', mat.get('output')).astype(np.float32)
        if u.ndim == 3:
            u = u[:, -1, :]

    a, u = torch.from_numpy(a), torch.from_numpy(u)

    N_orig = a.shape[-1]
    if cfg.resolution < N_orig:
        step = N_orig // cfg.resolution
        a = a[:, ::step][:, :cfg.resolution]
        u = u[:, ::step][:, :cfg.resolution]

    n_train = min(cfg.n_train, a.shape[0] - cfg.n_test)

    return {
        'a_train': a[:n_train].to(device),
        'u_train': u[:n_train].to(device),
        'a_test': a[n_train:n_train + cfg.n_test].to(device),
        'u_test': u[n_train:n_train + cfg.n_test].to(device),
        'N': a.shape[-1],
    }


def load_darcy(cfg: SpectraConfig, device: torch.device, data_dir: str = './data', use_normalizer: bool = True) -> Dict:
    """Load Darcy flow data."""
    os.makedirs(data_dir, exist_ok=True)

    train_path = f'{data_dir}/piececonst_r421_N1024_smooth1.mat'
    test_path = f'{data_dir}/piececonst_r421_N1024_smooth2.mat'

    if not os.path.exists(train_path) and GDRIVE_IDS.get('darcy_421'):
        print("Downloading Darcy data...")
        download_gdrive(GDRIVE_IDS['darcy_421'], f'{data_dir}/darcy.mat')
        # May need to handle zip extraction here

    # Try loading
    try:
        import h5py
        with h5py.File(train_path, 'r') as f:
            a_train = np.array(f['coeff']).T.astype(np.float32)
            u_train = np.array(f['sol']).T.astype(np.float32)
        with h5py.File(test_path, 'r') as f:
            a_test = np.array(f['coeff']).T.astype(np.float32)
            u_test = np.array(f['sol']).T.astype(np.float32)
    except:
        mat = scipy.io.loadmat(train_path)
        a_train, u_train = mat['coeff'].astype(np.float32), mat['sol'].astype(np.float32)
        mat = scipy.io.loadmat(test_path)
        a_test, u_test = mat['coeff'].astype(np.float32), mat['sol'].astype(np.float32)

    # Downsample 421 → 85
    s = 5
    a_train = torch.from_numpy(a_train[:cfg.n_train, ::s, ::s])
    u_train = torch.from_numpy(u_train[:cfg.n_train, ::s, ::s])
    a_test = torch.from_numpy(a_test[:cfg.n_test, ::s, ::s])
    u_test = torch.from_numpy(u_test[:cfg.n_test, ::s, ::s])

    H, W = a_train.shape[1], a_train.shape[2]

    data = {
        'a_train': a_train.to(device), 'u_train': u_train.to(device),
        'a_test': a_test.to(device), 'u_test': u_test.to(device),
        'H': H, 'W': W, 'y_normalizer': None,
    }

    if use_normalizer:
        y_norm = UnitGaussianNormalizer(u_train).to(device)
        data['y_normalizer'] = y_norm
        data['u_train_norm'] = y_norm.encode(data['u_train'])

    return data


def load_navier_stokes(cfg: SpectraConfig, device: torch.device, variant: str = 'v1e3',
                       T_in: int = 10, T_out: int = 10, data_dir: str = './data') -> Dict:
    """Load Navier-Stokes vorticity data for autoregressive training."""
    os.makedirs(data_dir, exist_ok=True)

    if variant == 'v1e3':
        patterns = [f'{data_dir}/ns_V1e-3*.mat', f'{data_dir}/NavierStokes_V1e-3*.mat', f'{data_dir}/*V1e-3*.mat']
        gdrive_key = 'ns_v1e3'
    else:
        patterns = [f'{data_dir}/NavierStokes_V1e-5*.mat', f'{data_dir}/*V1e-5*.mat']
        gdrive_key = 'ns_v1e5'

    filepath = None
    for pattern in patterns:
        files = glob.glob(pattern)
        if files:
            filepath = files[0]
            break

    if filepath is None and GDRIVE_IDS.get(gdrive_key):
        print(f"Downloading NS {variant} data...")
        filepath = f'{data_dir}/ns_{variant}.mat'
        download_gdrive(GDRIVE_IDS[gdrive_key], filepath)

    if filepath is None:
        raise FileNotFoundError(f"Could not find NS {variant} data")

    # Load
    try:
        import h5py
        with h5py.File(filepath, 'r') as f:
            keys = list(f.keys())
            data = np.array(f.get('u', f.get('vorticity', f[keys[0]]))).astype(np.float32)
    except:
        mat = scipy.io.loadmat(filepath)
        keys = [k for k in mat.keys() if not k.startswith('_')]
        data = mat.get('u', mat.get('vorticity', mat[keys[0]])).astype(np.float32)

    # Reshape to [N, T, H, W]
    if data.ndim == 4:
        dims = data.shape
        if dims[-1] < dims[1] and dims[-1] < 100:
            data = np.transpose(data, (0, 3, 1, 2))
        elif dims[0] < 100 and dims[-1] > 1000:
            data = np.transpose(data, (3, 0, 1, 2))

    N_total, T_total, H_orig, W_orig = data.shape

    # Subsample spatial
    resolution = cfg.resolution or 64
    if resolution < H_orig:
        step = H_orig // resolution
        data = data[:, :, ::step, ::step][:, :, :resolution, :resolution]

    H, W = resolution, resolution

    if T_in + T_out > T_total:
        T_out = T_total - T_in

    a = torch.from_numpy(data[:, :T_in])
    u = torch.from_numpy(data[:, T_in:T_in + T_out])

    n_train = min(cfg.n_train, a.shape[0] - cfg.n_test)
    n_test = min(cfg.n_test, a.shape[0] - n_train)

    return {
        'a_train': a[:n_train].to(device), 'u_train': u[:n_train].to(device),
        'a_test': a[n_train:n_train + n_test].to(device), 'u_test': u[n_train:n_train + n_test].to(device),
        'H': H, 'W': W, 'T_in': T_in, 'T_out': T_out,
    }


def load_data(cfg: SpectraConfig, device: torch.device, data_dir: str = './data') -> Dict:
    """Load data based on config."""
    dataset = cfg.dataset.lower()

    if dataset == 'burgers_r10':
        return load_burgers(cfg, device, data_dir)
    elif dataset == 'burgers_r1000':
        return load_burgers_r1000(cfg, device, data_dir)
    elif dataset == 'darcy':
        return load_darcy(cfg, device, data_dir)
    elif dataset in ['ns_v1e3', 'ns_v1e5']:
        variant = 'v1e3' if 'v1e3' in dataset else 'v1e5'
        return load_navier_stokes(cfg, device, variant, data_dir=data_dir)
    else:
        raise ValueError(f"Unknown dataset: {dataset}")


# ══════════════════════════════════════════════════════════════════════════════════════════

In [41]:
# @title 18. TRAINING
# ══════════════════════════════════════════════════════════════════════════════════════════
def get_optimizer(model: nn.Module, cfg: SpectraConfig) -> torch.optim.Optimizer:
    """Get optimizer with separate parameter groups for beta and h_eff."""
    if hasattr(model, 'beta_parameters') and len(list(model.beta_parameters())) > 0:
        beta_params = list(model.beta_parameters())
        heff_params = list(model.heff_parameters())
        special_ids = set(id(p) for p in beta_params + heff_params)
        backbone_params = [p for p in model.parameters() if id(p) not in special_ids]

        return torch.optim.AdamW([
            {'params': backbone_params, 'lr': cfg.lr, 'weight_decay': cfg.weight_decay},
            {'params': beta_params, 'lr': cfg.lr_beta, 'weight_decay': 0.0},
            {'params': heff_params, 'lr': cfg.lr_heff, 'weight_decay': 0.0},
        ])
    return torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)


def get_scheduler(optimizer: torch.optim.Optimizer, cfg: SpectraConfig, steps_per_epoch: int):
    """Get learning rate scheduler."""
    if cfg.scheduler == 'onecycle':
        return torch.optim.lr_scheduler.OneCycleLR(
            optimizer, max_lr=cfg.lr, epochs=cfg.epochs, steps_per_epoch=steps_per_epoch,
            pct_start=cfg.pct_start, div_factor=25.0, final_div_factor=1e4)
    elif cfg.scheduler == 'step':
        return torch.optim.lr_scheduler.StepLR(optimizer, step_size=cfg.step_size, gamma=cfg.gamma)
    elif cfg.scheduler == 'cosine':
        return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, cfg.epochs, eta_min=1e-6)
    raise ValueError(f"Unknown scheduler: {cfg.scheduler}")


def train_1d(model: nn.Module, data: Dict, cfg: SpectraConfig, device: torch.device, name: str = '') -> Dict:
    """Train 1D model."""
    model = model.to(device)
    a_train, u_train = data['a_train'], data['u_train']
    a_test, u_test = data['a_test'], data['u_test']

    steps_per_epoch = max(1, len(a_train) // cfg.batch_size)
    is_onecycle = cfg.scheduler == 'onecycle'

    optimizer = get_optimizer(model, cfg)
    scheduler = get_scheduler(optimizer, cfg, steps_per_epoch)

    print(f"\n{'─'*60}\n{name} | {count_params(model)/1e3:.1f}K params\n{'─'*60}")

    best_loss = float('inf')
    t0 = time.time()

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        perm = torch.randperm(len(a_train), device=device)
        epoch_loss, n_batches = 0.0, 0

        for i in range(0, len(a_train), cfg.batch_size):
            idx = perm[i:i + cfg.batch_size]
            pred = model(a_train[idx])
            loss = rel_l2_loss(pred, u_train[idx])

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            optimizer.step()

            if is_onecycle:
                scheduler.step()

            epoch_loss += loss.item()
            n_batches += 1

        if not is_onecycle:
            scheduler.step()

        train_loss = epoch_loss / n_batches

        if epoch % cfg.eval_every == 0 or epoch == 1 or epoch == cfg.epochs:
            model.eval()
            with torch.no_grad():
                test_loss = rel_l2_loss(model(a_test), u_test).item()

            is_best = test_loss < best_loss
            best_loss = min(best_loss, test_loss)

            params = model.get_params() if hasattr(model, 'get_params') else {}
            star = " ★" if is_best else ""
            param_str = " | " + ", ".join([f"β={b:.2f}" for b in params.get('betas', [])[:4]]) if params else ""
            print(f"  Ep {epoch:4d}: Train={100*train_loss:.3f}% Test={100*test_loss:.3f}% ({time.time()-t0:.0f}s){star}{param_str}")

    return {'best_loss': best_loss, 'params': model.get_params() if hasattr(model, 'get_params') else {}}


def train_2d(model: nn.Module, data: Dict, cfg: SpectraConfig, device: torch.device, name: str = '') -> Dict:
    """Train 2D model with optional normalization."""
    model = model.to(device)
    a_train, u_train = data['a_train'], data['u_train']
    a_test, u_test = data['a_test'], data['u_test']

    y_norm = data.get('y_normalizer')
    u_train_target = data.get('u_train_norm', u_train) if y_norm else u_train

    steps_per_epoch = max(1, len(a_train) // cfg.batch_size)
    is_onecycle = cfg.scheduler == 'onecycle'

    optimizer = get_optimizer(model, cfg)
    scheduler = get_scheduler(optimizer, cfg, steps_per_epoch)

    norm_str = " [normalized]" if y_norm else ""
    print(f"\n{'─'*60}\n{name} | {count_params(model)/1e3:.1f}K params{norm_str}\n{'─'*60}")

    best_loss = float('inf')
    t0 = time.time()

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        perm = torch.randperm(len(a_train), device=device)
        epoch_loss, n_batches = 0.0, 0

        for i in range(0, len(a_train), cfg.batch_size):
            idx = perm[i:i + cfg.batch_size]
            pred = model(a_train[idx])
            loss = rel_l2_loss(pred, u_train_target[idx])

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            optimizer.step()

            if is_onecycle:
                scheduler.step()

            epoch_loss += loss.item()
            n_batches += 1

        if not is_onecycle:
            scheduler.step()

        train_loss = epoch_loss / n_batches

        if epoch % cfg.eval_every == 0 or epoch == 1 or epoch == cfg.epochs:
            model.eval()
            with torch.no_grad():
                pred_test = model(a_test)
                if y_norm:
                    pred_test = y_norm.decode(pred_test)
                test_loss = rel_l2_loss(pred_test, u_test).item()

            is_best = test_loss < best_loss
            best_loss = min(best_loss, test_loss)

            params = model.get_params() if hasattr(model, 'get_params') else {}
            star = " ★" if is_best else ""
            param_str = " | " + ", ".join([f"β={b:.2f}" for b in params.get('betas', [])[:4]]) if params else ""
            print(f"  Ep {epoch:4d}: Train={100*train_loss:.3f}% Test={100*test_loss:.3f}% ({time.time()-t0:.0f}s){star}{param_str}")

    return {'best_loss': best_loss, 'params': model.get_params() if hasattr(model, 'get_params') else {}}


def train_2d_ar(model: nn.Module, data: Dict, cfg: SpectraConfig, device: torch.device,
                name: str = '', T_in: int = 10, T_out: int = 10) -> Dict:
    """Train 2D model with autoregressive single-step prediction."""
    model = model.to(device)

    a_train, u_train = data['a_train'], data['u_train']
    a_test, u_test = data['a_test'], data['u_test']

    data_train = torch.cat([a_train, u_train], dim=1)
    data_test = torch.cat([a_test, u_test], dim=1)

    T_total = data_train.shape[1]
    n_pairs = T_total - T_in

    steps_per_epoch = max(1, len(data_train) * n_pairs // cfg.batch_size)
    is_onecycle = cfg.scheduler == 'onecycle'

    optimizer = get_optimizer(model, cfg)
    scheduler = get_scheduler(optimizer, cfg, steps_per_epoch)

    print(f"\n{'─'*60}\n{name} | {count_params(model)/1e3:.1f}K | AR: T_in={T_in}→rollout {T_out}\n{'─'*60}")

    best_loss = float('inf')
    t0 = time.time()

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        perm = torch.randperm(len(data_train), device=device)
        epoch_loss, n_batches = 0.0, 0

        for i in range(0, len(data_train), cfg.batch_size):
            idx = perm[i:i + cfg.batch_size]
            batch = data_train[idx]
            B_actual = len(idx)

            t_starts = torch.randint(0, n_pairs, (B_actual,), device=device)
            histories = torch.stack([batch[b, t:t+T_in] for b, t in enumerate(t_starts)])
            targets = torch.stack([batch[b, t+T_in] for b, t in enumerate(t_starts)])

            pred = model(histories)
            loss = rel_l2_loss(pred, targets)

            if torch.isnan(loss):
                return {'best_loss': float('inf'), 'params': {}}

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            optimizer.step()

            if is_onecycle:
                scheduler.step()

            epoch_loss += loss.item()
            n_batches += 1

        if not is_onecycle:
            scheduler.step()

        train_loss = epoch_loss / max(n_batches, 1)

        if epoch % cfg.eval_every == 0 or epoch == 1 or epoch == cfg.epochs:
            model.eval()
            with torch.no_grad():
                # Autoregressive rollout
                current = a_test.clone()
                preds = []
                for _ in range(T_out):
                    next_step = model(current)
                    preds.append(next_step)
                    current = torch.cat([current[:, 1:], next_step.unsqueeze(1)], dim=1)
                pred_traj = torch.stack(preds, dim=1)
                test_loss = rel_l2_loss(pred_traj, u_test).item()

            is_best = test_loss < best_loss
            best_loss = min(best_loss, test_loss)

            params = model.get_params() if hasattr(model, 'get_params') else {}
            star = " ★" if is_best else ""
            param_str = " | " + ", ".join([f"β={b:.2f}" for b in params.get('betas', [])[:4]]) if params else ""
            print(f"  Ep {epoch:4d}: Train={100*train_loss:.3f}% Test={100*test_loss:.3f}% ({time.time()-t0:.0f}s){star}{param_str}")

    return {'best_loss': best_loss, 'params': model.get_params() if hasattr(model, 'get_params') else {}}


# ══════════════════════════════════════════════════════════════════════════════════════════

In [42]:
# @title 19. EXPERIMENT RUNNER
# ══════════════════════════════════════════════════════════════════════════════════════════
def run_experiment(dataset: str, model_type: str, preset: str, use_del: bool,
                   device: torch.device, data_dir: str = './data') -> Dict:
    """
    Run a single training experiment.

    Args:
        dataset: 'burgers_r10', 'burgers_r1000', 'darcy', 'ns_v1e3', 'ns_v1e5'
        model_type: 'fno' or 'transolver'
        preset: 'original', 'thuml', 'custom'
        use_del: Whether to use DEL-enhanced model
        device: torch device
        data_dir: Directory containing data files

    Returns:
        Dict with 'best_loss', 'params', 'config'
    """
    cfg = get_config(dataset, model_type, preset)
    set_seed(cfg.seed)

    # Determine experiment type
    is_1d = 'burgers' in dataset
    is_temporal = 'ns' in dataset

    # Create model
    model = create_model(cfg, use_del=use_del)
    del_str = "DEL-" if use_del else ""
    name = f"{del_str}{model_type.upper()} [{dataset}]"

    # Load data
    data = load_data(cfg, device, data_dir)

    # Train
    if is_1d:
        result = train_1d(model, data, cfg, device, name)
    elif is_temporal:
        T_in = data.get('T_in', 10)
        T_out = data.get('T_out', 10)
        result = train_2d_ar(model, data, cfg, device, name, T_in, T_out)
    else:
        result = train_2d(model, data, cfg, device, name)

    result['config'] = cfg
    result['name'] = name
    return result


def run_fno_experiments(preset: str = 'thuml', data_dir: str = './data'):
    """
    Run all FNO experiments.

    Datasets: burgers_r10, burgers_r1000, darcy, ns_v1e3, ns_v1e5
    Models: FNO, DEL-FNO
    """
    print("\n" + "═"*80)
    print("  FNO EXPERIMENTS")
    print("═"*80)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\nDevice: {device}")
    print(f"Preset: {preset}")
    print(f"Data directory: {data_dir}")

    datasets = ['burgers_r10', 'ns_v1e5']
    results = []

    for dataset in datasets:
        print(f"\n{'━'*80}")
        print(f"  Dataset: {dataset.upper()}")
        print(f"{'━'*80}")

        # Run baseline FNO
        result_base = run_experiment(dataset, 'fno', preset, use_del=False,
                                     device=device, data_dir=data_dir)
        results.append(('FNO', dataset, result_base))

        # Run DEL-FNO
        result_del = run_experiment(dataset, 'fno', preset, use_del=True,
                                    device=device, data_dir=data_dir)
        results.append(('DEL-FNO', dataset, result_del))

    # Summary
    print("\n" + "═"*80)
    print("  FNO RESULTS SUMMARY")
    print("═"*80)
    print(f"\n{'Model':<15} {'Dataset':<15} {'Best Loss':<12} {'Learned βs'}")
    print("─"*70)
    for model_name, dataset, result in results:
        betas = result.get('params', {}).get('betas', [])
        beta_str = ', '.join([f'{b:.2f}' for b in betas[:4]]) if betas else 'N/A'
        print(f"{model_name:<15} {dataset:<15} {result['best_loss']:.6f}     {beta_str}")

    return results


def run_transolver_experiments(preset: str = 'thuml', data_dir: str = './data'):
    """
    Run all Transolver experiments.

    Datasets: burgers_r10, burgers_r1000, darcy, ns_v1e3, ns_v1e5
    Models: Transolver, DEL-Transolver
    """
    print("\n" + "═"*80)
    print("  TRANSOLVER EXPERIMENTS")
    print("═"*80)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"\nDevice: {device}")
    print(f"Preset: {preset}")
    print(f"Data directory: {data_dir}")

    datasets = ['burgers_r10', 'darcy']
    results = []

    for dataset in datasets:
        print(f"\n{'━'*80}")
        print(f"  Dataset: {dataset.upper()}")
        print(f"{'━'*80}")

        # Run baseline Transolver
        result_base = run_experiment(dataset, 'transolver', preset, use_del=False,
                                     device=device, data_dir=data_dir)
        results.append(('Transolver', dataset, result_base))

        # Run DEL-Transolver
        result_del = run_experiment(dataset, 'transolver', preset, use_del=True,
                                    device=device, data_dir=data_dir)
        results.append(('DEL-Transolver', dataset, result_del))

    # Summary
    print("\n" + "═"*80)
    print("  TRANSOLVER RESULTS SUMMARY")
    print("═"*80)
    print(f"\n{'Model':<18} {'Dataset':<15} {'Best Loss':<12} {'Learned βs'}")
    print("─"*75)
    for model_name, dataset, result in results:
        betas = result.get('params', {}).get('betas', [])
        beta_str = ', '.join([f'{b:.2f}' for b in betas[:4]]) if betas else 'N/A'
        print(f"{model_name:<18} {dataset:<15} {result['best_loss']:.6f}     {beta_str}")

    return results


def run_all_experiments(preset: str = 'thuml', data_dir: str = './data'):
    """
    Run complete DELNO-Spectra benchmark.

    Section 1: FNO experiments
    Section 2: Transolver experiments
    """
    print("\n" + "╔" + "═"*78 + "╗")
    print("║" + " "*20 + "DELNO-Spectra v7: Complete Benchmark" + " "*21 + "║")
    print("║" + " "*20 + "ICML 2026 Submission" + " "*38 + "║")
    print("╚" + "═"*78 + "╝")

    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"\nStarted: {timestamp}")
    print(f"Preset: {preset}")
    print(f"Data directory: {data_dir}")

    all_results = {}

    # Section 1: FNO
    fno_results = run_fno_experiments(preset=preset, data_dir=data_dir)
    all_results['fno'] = fno_results

    # Section 2: Transolver
    transolver_results = run_transolver_experiments(preset=preset, data_dir=data_dir)
    all_results['transolver'] = transolver_results

    # Final Summary
    print("\n" + "╔" + "═"*78 + "╗")
    print("║" + " "*25 + "FINAL RESULTS SUMMARY" + " "*32 + "║")
    print("╚" + "═"*78 + "╝")

    print(f"\n{'Model':<18} {'Dataset':<15} {'Baseline':<12} {'DEL':<12} {'Δ':<10}")
    print("─"*75)

    # Group by dataset for comparison
    for model_type in ['fno', 'transolver']:
        results = all_results[model_type]
        model_name = 'FNO' if model_type == 'fno' else 'Transolver'

        for dataset in ['burgers_r100', 'burgers_r1000', 'darcy', 'ns_v1e3', 'ns_v1e5']:
            base = next((r for m, d, r in results if m == model_name and d == dataset), None)
            del_model = next((r for m, d, r in results if 'DEL' in m and d == dataset), None)

            if base and del_model:
                delta = (base['best_loss'] - del_model['best_loss']) / base['best_loss'] * 100
                delta_str = f"{delta:+.2f}%" if delta != 0 else "0.00%"
                print(f"{model_name:<18} {dataset:<15} {base['best_loss']:.6f}     {del_model['best_loss']:.6f}     {delta_str}")

    print("\n" + "═"*80)
    print(f"Completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("═"*80)

    return all_results


# ══════════════════════════════════════════════════════════════════════════════════════════

---
# FNO Experiments
Run FNO and DEL-FNO

In [43]:
fno_results = run_fno_experiments(preset='thuml', data_dir='./data')


════════════════════════════════════════════════════════════════════════════════
  FNO EXPERIMENTS
════════════════════════════════════════════════════════════════════════════════

Device: cuda
Preset: thuml
Data directory: ./data

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Dataset: BURGERS_R10
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

────────────────────────────────────────────────────────────
FNO [burgers_r10] | 221.9K params
────────────────────────────────────────────────────────────
  Ep    1: Train=92.408% Test=64.432% (3s) ★
  Ep  100: Train=2.148% Test=3.001% (100s) ★
  Ep  200: Train=1.508% Test=1.685% (192s) ★
  Ep  300: Train=0.788% Test=0.884% (283s) ★
  Ep  400: Train=0.240% Test=0.218% (374s) ★
  Ep  500: Train=0.038% Test=0.098% (465s) ★

────────────────────────────────────────────────────────────
DEL-FNO [burgers_r10] | 222.0K params
───────────────────────────────────────────────────────

---
# Transolver Experiments
Run Transolver and DEL-Transolver on all 5 datasets

In [ ]:
transolver_results = run_transolver_experiments(preset='thuml', data_dir='./data')


════════════════════════════════════════════════════════════════════════════════
  TRANSOLVER EXPERIMENTS
════════════════════════════════════════════════════════════════════════════════

Device: cuda
Preset: thuml
Data directory: ./data

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Dataset: BURGERS_R10
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

────────────────────────────────────────────────────────────
TRANSOLVER [burgers_r10] | 959.7K params
────────────────────────────────────────────────────────────
  Ep    1: Train=46.902% Test=40.673% (4s) ★
  Ep  100: Train=8.125% Test=7.195% (403s) ★
  Ep  200: Train=4.153% Test=4.114% (806s) ★
  Ep  300: Train=1.806% Test=2.195% (1209s) ★
  Ep  400: Train=0.618% Test=0.934% (1613s) ★
  Ep  500: Train=0.150% Test=0.654% (2016s) ★

────────────────────────────────────────────────────────────
DEL-TRANSOLVER [burgers_r10] | 959.9K params
───────────────────────────────